In [ ]:
# === CARGA DE CONFIGURACIÓN DEL PROYECTO ===
from config_proyecto import (
    BASE_DIR, TIF_DIR, CSV_CLIMA, CSV_STATS,
    OUTPUT_DIR, REPORT_DIR, ESCALA_TEMPORAL,
    N_CLUSTERS, CRS_OBJETIVO
)

print(f"📁 Base del proyecto: {BASE_DIR}")
print(f"🗂️ Imágenes: {TIF_DIR}")
print(f"📄 Clima: {CSV_CLIMA}")
print(f"📄 Estadísticas espectrales: {CSV_STATS}")


# 🛰️ Análisis de NDVI y Clima para Cultivo de Maní
Este notebook ejecuta el análisis completo de imágenes Sentinel-2, estadísticas de índices e integración con clima para el área de Manfredi (Córdoba).

In [ ]:
import sys
import os

# Asegura que el notebook vea los módulos del proyecto
PROJECT_PATH = os.getcwd()
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from preprocessing import load_rasters_from_folder
from classification import classify_images_kmeans
from stats import compute_area_per_class, build_area_time_series
from clima import (
    load_climate_csv,
    merge_index_climate,
    compute_correlations,
    train_rf_regressor
)
from visualization import (
    plot_area_time_series,
    plot_class_map,
)
from report_generator import create_report

# Rutas
DATA_DIR = r"C:/Users/sdari/Desktop/MANI_CORDOBA"
CSV_CLIMA = os.path.join(DATA_DIR, "datos/externos/Datos_Manfredi.csv")
CSV_STATS = os.path.join(DATA_DIR, "datos/raw/estadisticas_mani_s2.csv")
OUTPUT_DIR = os.path.join(os.getcwd(), "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
tif_paths = sorted([os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.endswith(".tif")])
dates = [datetime.strptime(os.path.basename(f).split("_")[1], "%Y%m%d") for f in tif_paths]
image_stack = load_rasters_from_folder(tif_paths)
classified_stack, kmeans = classify_images_kmeans(image_stack, n_clusters=4)


In [ ]:
area_df = compute_area_per_class(classified_stack)
area_df["fecha"] = dates
ts_df = build_area_time_series(area_df)
plot_area_time_series(ts_df)
plt.savefig(os.path.join(OUTPUT_DIR, "area_series.png"), dpi=300)


In [ ]:
df_stats = pd.read_csv(CSV_STATS)
df_stats["fecha"] = pd.to_datetime(df_stats["fecha"], format="%Y%m%d")
df_stats.set_index("fecha", inplace=True)
serie_ndvi = df_stats["NDVI_mean"].dropna()

# Guardar serie NDVI
plt.figure(figsize=(10, 4))
serie_ndvi.plot(marker='o')
plt.title("Serie temporal de NDVI")
plt.ylabel("NDVI")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "ndvi_series.png"), dpi=300)
plt.close()


In [ ]:
df_clima = load_climate_csv(CSV_CLIMA)
df_merged = merge_index_climate(serie_ndvi, df_clima, lag_days=10)
correls = compute_correlations(df_merged)

plt.figure(figsize=(6, 5))
sns.heatmap(pd.DataFrame(df_merged).corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlación entre NDVI y clima")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "correlation_plot.png"), dpi=300)
plt.close()


In [ ]:
model, r2, rmse = train_rf_regressor(df_merged)

# Mostrar resumen
print(f"R² = {r2:.2f} | RMSE = {rmse:.2f}")


In [ ]:
from datetime import date

resumen = (
    f"Se analizaron {len(tif_paths)} fechas entre {min(dates).date()} y {max(dates).date()}.
"
    f"El modelo de Random Forest para NDVI explicó el {r2*100:.1f}% de la varianza (R²={r2:.2f}, RMSE={rmse:.2f}).\n"
    f"Las variables climáticas más correlacionadas fueron: "
    + ", ".join([f"{k} ({v:.2f})" for k, v in sorted(correls.items(), key=lambda i: abs(i[1]), reverse=True)[:3]])
    + "."
)

create_report(resumen, OUTPUT_DIR)
